# Módulo 7 · Redes Neuronales de Grafos (GNN)

## Objetivos
1. Representar datos relacionales como **grafos** (matriz de adyacencia, listas de aristas, atributos) y calcular con **NetworkX** los algoritmos clásicos: caminos, **centralidades**, **detección de comunidades**.
2. Entender el **paso de mensajes** (*message passing*) como generalización de la convolución a grafos, e implementar desde cero **GCN**, **GraphSAGE** y **GAT**.
3. Resolver las tres tareas canónicas: clasificación de **nodos**, predicción de **enlaces** y clasificación de **grafos**.
4. Conocer las limitaciones (sobre-suavizado, expresividad ≤ test de Weisfeiler-Lehman, heterofilia) y las tendencias 2026 (**Graph Transformers**, GNN + LLM / GraphRAG, grafos temporales).
5. Manejar **PyTorch Geometric (PyG)** para trabajo real.

## Mapa de lectura (capítulos verificados)
| Tema | Libro | Capítulo verificado |
|---|---|---|
| Teoría de grafos y conceptos | Needham & Hodler, *Graph Algorithms* | **Cap. 2** Graph Theory and Concepts |
| Plataformas (Spark GraphFrames, Neo4j) | Needham & Hodler | **Cap. 3** Graph Platforms and Processing |
| Caminos mínimos, búsqueda | Needham & Hodler | **Cap. 4** Pathfinding and Graph Search Algorithms |
| Centralidades (grado, cercanía, intermediación, PageRank) | Needham & Hodler | **Cap. 5** Centrality Algorithms |
| Comunidades (Louvain, label propagation, componentes) | Needham & Hodler | **Cap. 6** Community Detection Algorithms |
| Grafos + ML: extracción de rasgos conectados, predicción de enlaces | Needham & Hodler | **Cap. 8** Using Graph Algorithms to Enhance Machine Learning |

> **Nota de actualización (2026).** La malla original citaba solo "Needham & Hodler Cap. 3"; ese capítulo es sobre *plataformas*. Los algoritmos están en Cap. 4-6 y la conexión con ML en Cap. 8. Como el libro (2019) **no cubre GNNs**, se añade bibliografía primaria gratuita: Hamilton, *Graph Representation Learning* (2020), y la documentación/tutoriales de **PyTorch Geometric**. Se añaden Graph Transformers y GraphRAG, dominantes hoy.

> **Ruta de aprendizaje Intensiva en ML Aplicado y DL Moderno** · Notebook del **Módulo 7 · Redes Neuronales de Grafos**
> Licencia CC BY-NC-SA 4.0 · © 2025-2026 Alexander Saravia
>
> Cómo usar este notebook: léelo de arriba hacia abajo. Cada sección tiene (1) la **idea** explicada con palabras y fórmulas,
> (2) **código ejecutable** que la materializa desde cero o con la librería estándar de 2026, (3) **lectura guiada** con
> el capítulo *verificado* de la bibliografía y (4) **ejercicios**. Las celdas marcadas `# OPCIONAL` requieren dependencias
> pesadas (GPU, Hugging Face, Spark) y están protegidas con `try/except` para que el resto del notebook siga corriendo.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, networkx as nx, time
print("NetworkX", nx.__version__)
rng = np.random.default_rng(0)

## 1. Grafos y algoritmos clásicos con NetworkX (Needham & Hodler Cap. 2, 4-6)

Un grafo $G=(V,E)$ se representa con la **matriz de adyacencia** $A\in\{0,1\}^{n\times n}$ (o con pesos) y la **matriz de grados** $D=\mathrm{diag}(A\mathbf{1})$. Usamos el clásico *club de karate de Zachary* (34 nodos, 2 facciones reales tras una escisión).

**Centralidades** (Cap. 5) — "¿quién importa?":
* *Grado*: número de vecinos. *Cercanía*: inverso de la distancia media al resto. *Intermediación*: fracción de caminos mínimos que pasan por el nodo (puentes). **PageRank**: probabilidad estacionaria de un paseo aleatorio con teletransporte — $\mathbf{r} = \alpha\,A^\top D^{-1}\mathbf{r} + (1-\alpha)\frac{\mathbf{1}}{n}$.

**Comunidades** (Cap. 6) — "¿quién está con quién?": **Louvain/Leiden** maximizan la *modularidad* $Q=\frac{1}{2m}\sum_{ij}\big[A_{ij}-\frac{k_ik_j}{2m}\big]\delta(c_i,c_j)$; *label propagation* difunde etiquetas por mayoría (¡el mismo principio del semi-supervisado del Módulo 2 y de las GNN!).

In [ ]:
G = nx.karate_club_graph()
y_true = np.array([0 if G.nodes[i]["club"] == "Mr. Hi" else 1 for i in G.nodes])
n, m = G.number_of_nodes(), G.number_of_edges()
print(f"{n} nodos, {m} aristas, densidad {nx.density(G):.3f}, diámetro {nx.diameter(G)}")

cent = {"grado": nx.degree_centrality(G), "cercanía": nx.closeness_centrality(G),
        "intermediación": nx.betweenness_centrality(G), "PageRank": nx.pagerank(G, alpha=0.85)}
import pandas as pd
tabla = pd.DataFrame(cent); tabla["facción"] = y_true
display(tabla.sort_values("PageRank", ascending=False).head(6).round(3))

# PageRank a mano (potencia) para ver que es álgebra lineal
A = nx.to_numpy_array(G); P = A / A.sum(1, keepdims=True); r = np.ones(n) / n
for _ in range(100): r = 0.85 * P.T @ r + 0.15 / n
print("PageRank manual == networkx:", np.allclose(r, [cent["PageRank"][i] for i in G.nodes], atol=1e-6))

In [ ]:
comunidades = nx.community.louvain_communities(G, seed=0)
c_louvain = np.zeros(n, int)
for k, com in enumerate(comunidades):
    for v in com: c_louvain[v] = k
print(f"Louvain encontró {len(comunidades)} comunidades, modularidad = {nx.community.modularity(G, comunidades):.3f}")

pos = nx.spring_layout(G, seed=4)
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
nx.draw(G, pos, ax=axes[0], node_color=y_true, cmap="coolwarm", node_size=[3000 * cent["PageRank"][i] for i in G.nodes], with_labels=True, font_size=7)
axes[0].set_title("facción real (color) y PageRank (tamaño)")
nx.draw(G, pos, ax=axes[1], node_color=c_louvain, cmap="tab10", node_size=120); axes[1].set_title("comunidades Louvain (no supervisado)")
sp = dict(nx.single_source_shortest_path_length(G, 0))
nx.draw(G, pos, ax=axes[2], node_color=[sp[i] for i in G.nodes], cmap="viridis", node_size=120); axes[2].set_title("distancia desde el nodo 0 (BFS)")
plt.show()

## 2. Del algoritmo al aprendizaje: paso de mensajes

Los algoritmos anteriores son **fijos**. Una GNN **aprende** qué agregar. Cada capa actualiza la representación $\mathbf{h}_v$ de un nodo combinando la suya con las de sus vecinos $\mathcal N(v)$:

$$\mathbf{h}_v^{(l+1)} = \mathrm{UPDATE}\Big(\mathbf{h}_v^{(l)},\ \mathrm{AGGREGATE}\big(\{\mathbf{h}_u^{(l)}: u\in\mathcal N(v)\}\big)\Big).$$

La agregación debe ser **invariante a permutaciones** (suma, media, máximo, atención). Tras $L$ capas, cada nodo "ve" su vecindario a $L$ saltos: la analogía exacta del campo receptivo de una CNN, donde la rejilla de píxeles es un grafo regular.

Tres agregadores clásicos:

| Modelo | Regla de capa | Idea |
|---|---|---|
| **GCN** (Kipf & Welling, 2017) | $H^{(l+1)} = \sigma\big(\tilde D^{-1/2}\tilde A\tilde D^{-1/2} H^{(l)} W^{(l)}\big)$, $\tilde A = A+I$ | media normalizada simétrica; filtro pasa-bajos espectral |
| **GraphSAGE** (Hamilton et al., 2017) | $\mathbf{h}_v \leftarrow \sigma\big(W\,[\mathbf{h}_v \,\|\, \mathrm{mean}_{u\in\mathcal N(v)}\mathbf{h}_u]\big)$ | separa "yo" de "vecinos"; muestreo de vecinos → escala a grafos enormes; **inductivo** |
| **GAT** (Veličković et al., 2018) | $\alpha_{vu} = \mathrm{softmax}_u\big(\mathrm{LeakyReLU}(\mathbf{a}^\top[W\mathbf{h}_v\|W\mathbf{h}_u])\big)$, $\mathbf{h}_v\leftarrow\sigma(\sum_u\alpha_{vu}W\mathbf{h}_u)$ | pesos aprendidos por arista (atención, Módulo 5) |

Los implementamos con matrices densas (suficiente para grafos de miles de nodos; PyG usa dispersas y *scatter*).

In [ ]:
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    TORCH = True; torch.manual_seed(0)
    print("PyTorch", torch.__version__)
except ImportError:
    TORCH = False; print("PyTorch no instalado; las secciones 2-4 lo requieren.")

if TORCH:
    A_t = torch.tensor(A, dtype=torch.float32); I = torch.eye(n)
    A_hat = A_t + I; d_hat = A_hat.sum(1); A_norm = A_hat / torch.sqrt(d_hat[:, None] * d_hat[None, :])   # D^-1/2 (A+I) D^-1/2

    class GCNLayer(nn.Module):
        def __init__(self, i, o): super().__init__(); self.lin = nn.Linear(i, o)
        def forward(self, H, A_norm): return A_norm @ self.lin(H)

    class SAGELayer(nn.Module):
        def __init__(self, i, o): super().__init__(); self.lin = nn.Linear(2 * i, o)
        def forward(self, H, A):
            mean_nb = (A @ H) / A.sum(1, keepdim=True).clamp(min=1)
            return self.lin(torch.cat([H, mean_nb], 1))

    class GATLayer(nn.Module):
        def __init__(self, i, o, heads=4):
            super().__init__(); self.h, self.o = heads, o
            self.W = nn.Linear(i, heads * o, bias=False); self.a_src = nn.Parameter(torch.randn(heads, o) * 0.1); self.a_dst = nn.Parameter(torch.randn(heads, o) * 0.1)
        def forward(self, H, A, return_att=False):
            Wh = self.W(H).view(-1, self.h, self.o)                                   # (n, heads, o)
            e = (Wh * self.a_dst).sum(-1)[:, None, :] + (Wh * self.a_src).sum(-1)[None, :, :]   # (n_dst, n_src, heads)
            e = F.leaky_relu(e, 0.2).masked_fill((A + torch.eye(len(A)))[:, :, None] == 0, float("-inf"))
            att = e.softmax(1)                                                          # normaliza sobre vecinos (incluye self-loop)
            out = torch.einsum("ijh,jho->iho", att, Wh).mean(1)                        # promedio de cabezas
            return (out, att) if return_att else out

    class GNN(nn.Module):
        def __init__(self, capa, d_in, d_h, d_out, **kw):
            super().__init__(); self.l1 = capa(d_in, d_h, **kw); self.l2 = capa(d_h, d_out, **kw); self.tipo = capa
        def forward(self, X, A, A_norm):
            op = A_norm if self.tipo is GCNLayer else A
            return self.l2(F.dropout(F.relu(self.l1(X, op)), 0.5, self.training), op)

## 3. Clasificación de nodos semi-supervisada

Escenario típico (Kipf & Welling): conocemos la etiqueta de **muy pocos** nodos (aquí 2: un líder de cada facción) y queremos etiquetar el resto. Sin atributos de nodo, usamos la identidad ($X=I$): la GNN aprende *embeddings* solo a partir de la estructura. Comparamos GCN, GraphSAGE y GAT contra una **regresión logística sobre atributos estructurales a mano** (centralidades + comunidad: el enfoque "conectado" de Needham & Hodler Cap. 8).

In [ ]:
if TORCH:
    X_id = torch.eye(n); y_t = torch.tensor(y_true)
    train_mask = torch.zeros(n, dtype=torch.bool); train_mask[[0, 33]] = True     # el instructor (Mr. Hi) y el administrador

    def entrenar_gnn(model, epochs=200, lr=0.01, wd=5e-4):
        opt = torch.optim.Adam(model.parameters(), lr, weight_decay=wd)
        for _ in range(epochs):
            model.train(); opt.zero_grad(); out = model(X_id, A_t, A_norm)
            F.cross_entropy(out[train_mask], y_t[train_mask]).backward(); opt.step()
        model.eval()
        with torch.no_grad(): pred = model(X_id, A_t, A_norm).argmax(1)
        return (pred[~train_mask] == y_t[~train_mask]).float().mean().item()

    resultados = {}
    for nombre, capa, kw in [("GCN", GCNLayer, {}), ("GraphSAGE", SAGELayer, {}), ("GAT", GATLayer, {"heads": 4})]:
        accs = [entrenar_gnn(GNN(capa, n, 16, 2, **kw)) for _ in range(5)]
        resultados[nombre] = f"{np.mean(accs):.3f} ± {np.std(accs):.3f}"

    # baseline: rasgos estructurales a mano + regresión logística (Needham & Hodler Cap. 8)
    from sklearn.linear_model import LogisticRegression
    F_hand = np.c_[tabla[["grado", "cercanía", "intermediación", "PageRank"]].values, np.eye(len(comunidades))[c_louvain]]
    lr_ = LogisticRegression().fit(F_hand[train_mask.numpy()], y_true[train_mask.numpy()])
    resultados["logística (rasgos a mano)"] = f"{lr_.score(F_hand[~train_mask.numpy()], y_true[~train_mask.numpy()]):.3f}"
    print("exactitud en los 32 nodos no etiquetados (con solo 2 etiquetas):")
    for k, v in resultados.items(): print(f"  {k:26s} {v}")

In [ ]:
if TORCH:
    # Visualizar los embeddings aprendidos por la GCN (capa oculta -> 2D) y la atención del GAT
    gcn = GNN(GCNLayer, n, 16, 2); entrenar_gnn(gcn)
    with torch.no_grad(): H = F.relu(gcn.l1(X_id, A_norm)).numpy()
    from sklearn.decomposition import PCA
    Z = PCA(2).fit_transform(H)
    gat = GNN(GATLayer, n, 16, 2, heads=4); entrenar_gnn(gat)
    with torch.no_grad(): _, att = gat.l2(F.relu(gat.l1(X_id, A_t)), A_t, return_att=True)
    att_mean = att.mean(-1).numpy()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(*Z.T, c=y_true, cmap="coolwarm", s=80, edgecolor="k"); [axes[0].annotate(i, Z[i], fontsize=7) for i in range(n)]
    axes[0].set_title("embeddings GCN (PCA 2D): la estructura separa las facciones")
    edges = [(u, v, att_mean[u, v]) for u, v in G.edges]
    nx.draw(G, pos, ax=axes[1], node_color=y_true, cmap="coolwarm", node_size=100,
            edge_color=[w for _, _, w in edges], edge_cmap=plt.cm.Greys, width=[1 + 6 * w for _, _, w in edges])
    axes[1].set_title("GAT: grosor = peso de atención aprendido por arista"); plt.show()

### Sobre-suavizado: por qué las GNN son poco profundas
Cada capa GCN aplica el operador $\hat A = \tilde D^{-1/2}\tilde A\tilde D^{-1/2}$, un **filtro pasa-bajos**. Al apilar $L$ capas, $\hat A^L X$ converge a un vector proporcional a $\tilde D^{1/2}\mathbf{1}$: **todos los nodos se vuelven iguales**. Por eso 2-3 capas suelen ser óptimas; para ir más profundo se usan residuales (GCNII), *jumping knowledge*, normalización o Graph Transformers. Lo medimos.

In [ ]:
if TORCH:
    Xr = torch.randn(n, 8); dist = []
    H = Xr.clone()
    for L in range(1, 41):
        H = A_norm @ H
        Hn = F.normalize(H, dim=1); dist.append(torch.cdist(Hn, Hn).mean().item())   # distancia media entre nodos
    plt.plot(range(1, 41), dist, "o-"); plt.xlabel("número de capas de propagación"); plt.ylabel("distancia media entre representaciones")
    plt.title("Sobre-suavizado: con muchas capas las representaciones colapsan"); plt.show()

## 4. Predicción de enlaces (Needham & Hodler Cap. 8)

¿Qué aristas *faltan* o *aparecerán*? Aplicaciones: recomendación (usuario–producto), descubrimiento de fármacos (proteína–compuesto), grafos de conocimiento. Enfoque **codificador–decodificador**: la GNN produce $\mathbf{z}_v$; la probabilidad de arista es $\sigma(\mathbf{z}_u^\top\mathbf{z}_v)$. Entrenamos con **muestreo negativo** (pares no conectados) y ocultamos un 15 % de aristas para evaluar (ROC-AUC). Comparamos con heurísticas clásicas: vecinos comunes, Adamic–Adar, Jaccard.

In [ ]:
if TORCH:
    from sklearn.metrics import roc_auc_score
    edges = np.array(G.edges); rng.shuffle(edges)
    n_test = int(0.15 * len(edges)); test_pos, train_pos = edges[:n_test], edges[n_test:]
    G_train = nx.Graph(); G_train.add_nodes_from(G.nodes); G_train.add_edges_from(train_pos)
    A_tr = torch.tensor(nx.to_numpy_array(G_train), dtype=torch.float32)
    A_tr_hat = A_tr + I; d_ = A_tr_hat.sum(1); A_tr_norm = A_tr_hat / torch.sqrt(d_[:, None] * d_[None, :])

    no_edges = np.array([(u, v) for u in range(n) for v in range(u + 1, n) if not G.has_edge(u, v)])
    test_neg = no_edges[rng.choice(len(no_edges), n_test, replace=False)]

    enc = GNN(GCNLayer, n, 32, 16); opt = torch.optim.Adam(enc.parameters(), 0.01)
    for ep in range(200):
        enc.train(); Z = enc(X_id, A_tr, A_tr_norm)
        neg = no_edges[rng.choice(len(no_edges), len(train_pos))]
        pos_s = (Z[train_pos[:, 0]] * Z[train_pos[:, 1]]).sum(1); neg_s = (Z[neg[:, 0]] * Z[neg[:, 1]]).sum(1)
        loss = F.binary_cross_entropy_with_logits(torch.cat([pos_s, neg_s]), torch.cat([torch.ones(len(pos_s)), torch.zeros(len(neg_s))]))
        opt.zero_grad(); loss.backward(); opt.step()
    enc.eval()
    with torch.no_grad():
        Z = enc(X_id, A_tr, A_tr_norm); pares = np.r_[test_pos, test_neg]; s_gnn = (Z[pares[:, 0]] * Z[pares[:, 1]]).sum(1).numpy()
    y_lp = np.r_[np.ones(n_test), np.zeros(n_test)]
    print(f"GCN encoder-decoder     ROC-AUC = {roc_auc_score(y_lp, s_gnn):.3f}")
    for nombre, fn in [("vecinos comunes", lambda u, v: len(list(nx.common_neighbors(G_train, u, v)))),
                       ("Adamic-Adar", lambda u, v: next(nx.adamic_adar_index(G_train, [(u, v)]))[2]),
                       ("Jaccard", lambda u, v: next(nx.jaccard_coefficient(G_train, [(u, v)]))[2])]:
        print(f"{nombre:24s} ROC-AUC = {roc_auc_score(y_lp, [fn(u, v) for u, v in pares]):.3f}")

## 5. Clasificación de grafos y expresividad

Para clasificar **grafos completos** (moléculas: ¿es tóxica?) se añade un **readout** invariante (suma/media de los $\mathbf{h}_v$) antes de la capa final. La **GIN** (Xu et al., 2019) usa agregación por *suma* + MLP y alcanza la expresividad máxima del paso de mensajes: la del **test de Weisfeiler–Lehman 1-dimensional**. Consecuencia: las GNN de paso de mensajes **no distinguen** ciertos grafos (p. ej., dos triángulos vs un hexágono, ambos 2-regulares). Lo comprobamos: una GCN con $X=\mathbf{1}$ produce embeddings idénticos para ambos.

In [ ]:
if TORCH:
    G1 = nx.disjoint_union(nx.cycle_graph(3), nx.cycle_graph(3)); G2 = nx.cycle_graph(6)   # ambos: 6 nodos, todos de grado 2
    def readout(Gx, layer):
        Ax = torch.tensor(nx.to_numpy_array(Gx), dtype=torch.float32); Ah = Ax + torch.eye(6); d = Ah.sum(1)
        An = Ah / torch.sqrt(d[:, None] * d[None, :]); H = torch.ones(6, 4)
        for _ in range(3): H = torch.tanh(layer(H, An))
        return H.sum(0)
    layer = GCNLayer(4, 4)
    r1, r2 = readout(G1, layer), readout(G2, layer)
    print("readout G1 (2 triángulos):", r1.detach().numpy().round(4)); print("readout G2 (hexágono)     :", r2.detach().numpy().round(4))
    print("indistinguibles por paso de mensajes (1-WL):", torch.allclose(r1, r2, atol=1e-5))
    fig, axes = plt.subplots(1, 2, figsize=(7, 3)); nx.draw(G1, ax=axes[0], node_size=80); nx.draw(G2, ax=axes[1], node_size=80); plt.show()
    print("Soluciones: rasgos posicionales/estructurales (Laplacian PE, conteo de ciclos), subgrafos, o Graph Transformers.")

## 6. PyTorch Geometric y el estado del arte (2026)

```python
# OPCIONAL — uv pip install torch_geometric
import torch_geometric as pyg
from torch_geometric.datasets import Planetoid, KarateClub
from torch_geometric.nn import GCNConv, SAGEConv, GATv2Conv, GINConv, global_mean_pool
from torch_geometric.loader import NeighborLoader                 # muestreo de vecinos para grafos gigantes

data = KarateClub()[0]                     # data.x, data.edge_index (2, E), data.y, data.train_mask
class Net(torch.nn.Module):
    def __init__(self):
        super().__init__(); self.c1 = GATv2Conv(data.num_features, 8, heads=4); self.c2 = GATv2Conv(32, 2)
    def forward(self, x, ei): return self.c2(F.relu(self.c1(x, ei)), ei)
```
Tendencias que debes conocer:
* **Graph Transformers** (Graphormer, GPS, Exphormer): atención global + codificaciones posicionales de grafo; superan a las GNN clásicas en moléculas y grafos medianos.
* **GNN + LLM / GraphRAG**: grafos de conocimiento construidos por LLM + recuperación sobre el grafo para preguntas "globales"; *embeddings* de texto como atributos de nodo.
* **Grafos temporales y heterogéneos** (TGN, HGT) para fraude, redes sociales, cadenas de suministro.
* **Escalado**: muestreo de vecinos (GraphSAGE), *cluster-GCN*, PyG + cuGraph/GPU; grafos de miles de millones de aristas en producción (Pinterest PinSage, Alibaba, Uber).
* **Modelos fundacionales de grafos** (2024-2026): investigación activa; aún sin un "GPT de grafos" consolidado.

## 📦 Recursos de los repositorios oficiales de los libros

| Libro | Recurso | Para qué usarlo aquí |
|---|---|---|
| Needham & Hodler, *Graph Algorithms* | Libro gratuito (cortesía de Neo4j): <https://neo4j.com/graph-algorithms-book/> · código de ejemplo: <https://resources.oreilly.com/examples/0636920233145> · la librería `neo4j-graph-algorithms` usada en el libro fue **sustituida por Neo4j Graph Data Science (GDS)**: <https://neo4j.com/docs/graph-data-science/current/> · repos de lectores que portan los ejemplos: [alexjolong/graph_algs_oreilly](https://github.com/alexjolong/graph_algs_oreilly), [trijini/Graph-Algorithms-...](https://github.com/trijini/Graph-Algorithms-Practical-Examples-in-Apache-Spark-and-Neo4j-Mark-Needham-Amy-E.-Hodler-) | Los ejemplos del Cap. 4-6 (transporte aéreo, redes sociales) se pueden reproducir con NetworkX; los del Cap. 8 (predicción de coautoría) con nuestro código de la sección 4. Si usas Neo4j, traduce las llamadas `algo.*` a `gds.*` |
| Hamilton, *Graph Representation Learning* (gratuito) | <https://www.cs.mcgill.ca/~wlh/grl_book/> · Cap. 5 (GNN), 6 (teoría), 7 (generativos) | Base teórica de las secciones 2-5 |
| PyTorch Geometric | [Tutoriales oficiales](https://pytorch-geometric.readthedocs.io/en/latest/get_started/colabs.html) (Colab: intro, node classification, graph classification, link prediction, GNN explanations) · [Stanford CS224W](https://web.stanford.edu/class/cs224w/) | Versión PyG de cada tarea de este notebook |

## 7. Ejercicios
1. **Centralidades vs facciones.** ¿Qué centralidad identifica mejor a los dos líderes (nodos 0 y 33)? ¿Y a los "puentes" entre facciones? Justifica con la definición.
2. **Atributos de nodo.** Sustituye $X=I$ por los rasgos estructurales `F_hand` como entrada a la GCN. ¿Mejora con 2 etiquetas? ¿Y con 8?
3. **Profundidad.** Entrena GCN con 1, 2, 4, 8 capas (añade residuales `H + capa(H)` en una variante). Grafica exactitud vs profundidad.
4. **Heterofilia.** Genera un grafo con `nx.stochastic_block_model` donde los nodos se conecten *más* con la otra clase que con la suya. ¿Sigue funcionando la GCN? Prueba GraphSAGE (separa "yo" de "vecinos").
5. **Predicción de enlaces mejorada.** Cambia el decodificador de producto punto a un MLP sobre $[\mathbf{z}_u\|\mathbf{z}_v\|\mathbf{z}_u\odot\mathbf{z}_v]$ y compara el AUC. Añade *hard negatives* (pares a distancia 2).
6. **GIN vs GCN.** Implementa una capa GIN ($\mathbf{h}_v \leftarrow \mathrm{MLP}((1+\epsilon)\mathbf{h}_v + \sum_u \mathbf{h}_u)$) y repite el experimento de los dos triángulos. Luego prueba con grafos que 1-WL sí distingue (p. ej., estrella vs camino).
7. **PyG real.** Instala `torch_geometric`, carga `Planetoid("Cora")` y reproduce el ~81 % de exactitud de la GCN original con `GCNConv`.

## Referencias
* Needham, M. & Hodler, A. E. (2019). *Graph Algorithms: Practical Examples in Apache Spark and Neo4j*. O'Reilly. Cap. 2-6, 8.
* Hamilton, W. L. (2020). *Graph Representation Learning*. Morgan & Claypool. Gratuito: https://www.cs.mcgill.ca/~wlh/grl_book/
* Kipf, T. & Welling, M. (2017). *Semi-Supervised Classification with Graph Convolutional Networks*. ICLR.
* Hamilton, W., Ying, R. & Leskovec, J. (2017). *Inductive Representation Learning on Large Graphs* (GraphSAGE). NeurIPS.
* Veličković, P. et al. (2018). *Graph Attention Networks*. ICLR. · Brody, S. et al. (2022). *How Attentive are GATs?* (GATv2). ICLR.
* Xu, K. et al. (2019). *How Powerful are Graph Neural Networks?* (GIN). ICLR.
* Rampášek, L. et al. (2022). *Recipe for a General, Powerful, Scalable Graph Transformer* (GPS). NeurIPS.
* Edge, D. et al. (2024). *From Local to Global: A Graph RAG Approach to Query-Focused Summarization*. Microsoft Research.
* PyTorch Geometric: https://pytorch-geometric.readthedocs.io · Stanford CS224W (Leskovec), material abierto.